[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/dec_stokes/dec_stokes.ipynb)

# Stokes Flow and Discrete Exterior Calculus

This example computes the slow viscous flow of a fluid through a channel past an offset circular obstacle, where inertia is negligible and pressure balances viscous resistance alone. The channel is modelled as a triangulated strip with a circumcentric dual: fluid crosses the edges as integrated fluxes, every triangle passes on exactly as much fluid as it receives, and the solid walls bring the fluid to rest.

Edge fluxes are scalar fields on the edges, and the incidences between vertices, edges and triangles carry only signs, so `d1(d0)` vanishes before any geometry enters. Joining a dual edge with the primal edge it crosses, `dual_edges & edges`, gives the oriented patch of their kite, and the diagonal Hodge maps scale by these patches, carrying the length ratios and the complementary orientation at once. Vorticity is a bivector at each vertex, `curl = inverse_star0(d0_vorticity.adjugate()(star1))`, and the viscous force is the same chain read back, `star1.adjugate()(d0_vorticity(curl))`, symmetric by construction.

In the matrix notation of discrete exterior calculus ([Desbrun et al., 2005](https://arxiv.org/abs/math/0508341)), as in the pressure–velocity formulation of incompressible flow by [Jagad et al.](https://arxiv.org/abs/2010.15520), the edge fluxes $u$ and face pressures $p$ solve the saddle-point system $\begin{pmatrix}\mu\,\star_1 d_0 \star_0^{-1} d_0^{T} \star_1 & -d_1^{T} \\ -d_1 & 0\end{pmatrix}\begin{pmatrix}u \\ p\end{pmatrix} = \begin{pmatrix}f \\ g\end{pmatrix}$, where each $\star_k$ is the unsigned diagonal of ratios $|\star\sigma^k|/|\sigma^k|$, and the vorticity $\omega = \star_0^{-1} d_0^{T} \star_1 u$ is one pseudoscalar per vertex whose sign follows a chosen orientation of the plane.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA2D as ga
from numga.sparse import spdiag
from examples.fluids.dec_stokes import render, scenarios
from examples.fluids.dec_stokes.numerical import stokes_solution
from examples.util.animation import save_figure
from numga.topology import Topology

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                # Scalar+
Bivector = ga.gatype.bivector()                            # Bivector+
Point = ga.gatype.antivector()                             # Antivector-
Patch = ga.gatype.antibivector()                           # Antibivector-

np.set_printoptions(precision=4, suppress=True)

## The channel

The obstacle leaves unequal gaps above and below it. A prescribed throughput divides between them, with the pressure drop determined by viscous resistance. Lengths are measured in half-channel widths, speeds in the mean inlet speed, and viscosity is one.

In [ ]:
n_divisions = 18
viscosity = 1.0
topology, vertices, inlet, outlet, walls = scenarios.channel(n_divisions)
V, E, F = topology.n_elements

save_figure(render.draw_triangulation(topology, vertices), "dec_stokes_triangulation");

## Cochains and incidence

A cochain holds one signed reading per simplex: vertex values, edge integrals or face integrals. Form degree belongs to the simplex grade, so each exterior derivative maps fields on one grade of simplices to fields on the next. Incidence contains only signs: `d1(d0)` vanishes because each vertex occurs twice with opposite signs in a triangle's boundary.

In [ ]:
d0 = topology.d[0] * Scalar                                # (Scalar+ <- Scalar+)[E <- V]
d1 = topology.d[1] * Scalar                                # (Scalar+ <- Scalar+)[F <- E]
d0_vorticity = topology.d[0] * Bivector                    # (Bivector+ <- Bivector+)[E <- V]

print("Vertex to edge:", d0.gatype, d0.shape)
print("Edge to face:  ", d1.gatype, d1.shape)

## Primal simplices and the circumcentric dual

Each triangle has a dual vertex at its circumcentre, where the distances to its corners agree. Neighbouring circumcentres define the dual edge crossing their shared primal edge. Around each primal vertex these dual edges enclose a polygonal dual cell.

In [ ]:
edges = topology.d[0] * vertices                           # Point-[E]
corners = topology.incidence[2, 0] * vertices              # [3] Point-[F]
sides = topology.boundary[2, 1] * edges                    # [3] Point-[F]
face_patches = (sides[0] & sides[1]) / 2                   # Patch-[F]
areas = face_patches.norm()                                # Scalar+[F]
face_centres = corners.sum(axis=0) / 3                     # Point-[F]

# On the paraboloid of half-squared distances, the plane through the three corners has slope equal to the circumcentre.
gradients = sides / (2 * face_patches)                     # [3] Vector+[F]
circumcentres = (gradients * (corners.norm_squared() / 2)).sum(axis=0).dual()  # Point-[F]

ones = mv.scalar(np.ones((F, 1))).field()                  # Scalar+[F]
midpoints = topology.incidence[1, 0].mean(axis=0) * vertices  # Point-[E]
boundary = topology.d[1].adjugate() * ones                       # Scalar+[E]
# Interior dual edges join adjacent circumcentres. At the boundary they
# finish at the primal edge's midpoint, closing the truncated dual cells.
# The dual edge crosses to the primal edge's right, matching positive flux.
dual_edges = boundary * midpoints - topology.d[1].adjugate() * circumcentres  # Point-[E]

save_figure(render.draw_complex(topology, vertices, circumcentres, dual_edges), "dec_stokes_complex");

The primal triangulation is blue and its dual is orange; the highlighted triangle and dual cell show the two kinds of area.

## Diagonal Hodge maps

Joining each dual edge with the primal edge it crosses gives the signed area of their kite as an intrinsic patch. Dividing by the squared primal edge length gives the dual-to-primal length ratio, and summing the quarter-kites around each vertex gives its dual cell patch. Multiplying by these oriented patches maps primal readings directly to dual patch readings, and dividing a dual cell's circulation by its area gives the vertex vorticity bivector.

In [ ]:
# Oriented dual/primal edge ratios and dual cell patches from the quarter-kites
# incident to each vertex.
edge_ratios = (dual_edges & edges) / edges.norm_squared()  # Patch-[E]
dual_patches = topology.incidence[1, 0].adjugate().sum(axis=0) * ((dual_edges & edges) / 4)  # Patch-[V]
dual_areas = dual_patches.norm()                           # Scalar+[V]

star0 = spdiag(dual_patches * Scalar)                      # (Patch- <- Scalar+)[V <- V]
star1 = spdiag(edge_ratios * Scalar)                       # (Patch- <- Scalar+)[E <- E]
star2 = spdiag(face_patches.inverse() * Scalar)            # (Patch- <- Scalar+)[F <- F]
inverse_star0 = Patch.dual_inverse() / dual_areas          # (Bivector+ <- Patch-)[V]

print("Primal to dual edge readings:", star1.gatype)
print("Dual circulation sum:        ", d0_vorticity.adjugate().gatype, d0_vorticity.adjugate().shape)

## Vorticity and viscous resistance

The dual of an edge flux is circulation along the crossing dual edge. A vorticity bivector pairs with a dual patch across `&`, so the adjugate of the vertex-to-edge incidence sums the circulation around each dual cell, and dividing by that dual cell's area leaves the vorticity at the vertex. Vorticity differences across primal edges, pulled back through the adjugate of the edge Hodge map, give the viscous force on each edge.

In [ ]:
curl = inverse_star0(d0_vorticity.adjugate()(star1))       # (Bivector+ <- Scalar+)[V <- E]
viscous = viscosity * star1.adjugate()(d0_vorticity(curl)) # (Antiscalar- <- Scalar+)[E <- E]

## Boundary flow and pressure

Pressure enforces zero net flux through every triangle. The inlet and outlet carry the same parabolic profile. On the solid walls, zero normal flux and zero tangential boundary circulation impose no slip. Fluxes and pressures are solved together, and the arbitrary pressure offset is fixed by its area-weighted mean.

In [ ]:
# The inlet and outlet carry the same parabolic flux; the solid walls carry none.
prescribed_flux = scenarios.boundary_flux(topology, vertices, inlet, outlet)  # Scalar+[E]
fixed_edges = topology.exterior[1]
boundary_circulation = dual_patches * 0                    # Patch-[V]

# Boundary segments close the dual cells; their prescribed circulation
# contributes to vorticity and hence to viscous resistance.
boundary_vorticity = inverse_star0(boundary_circulation)   # Bivector+[V]
boundary_resistance = viscosity * star1.adjugate()(d0_vorticity(boundary_vorticity))  # Antiscalar-[E]

free_edges = np.setdiff1d(np.arange(E), fixed_edges)
embedding = topology.reading(free_edges, 1).adjugate() * Scalar      # (Scalar+ <- Scalar+)[E <- E_free]
stiffness = embedding.adjugate()(viscous(embedding))       # (Antiscalar- <- Scalar+)[E_free <- E_free]
divergence = d1(embedding)                                 # (Scalar+ <- Scalar+)[F <- E_free]
force = -embedding.adjugate()(viscous(prescribed_flux) + boundary_resistance)  # Antiscalar-[E_free]
balance = -d1(prescribed_flux)                             # Scalar+[F]
free_flux, pressure = stokes_solution(stiffness, divergence, force, balance)  # Scalar+[E_free], Scalar+[F]
flux = prescribed_flux + embedding(free_flux)              # Scalar+[E]
pressure = pressure - (pressure * areas).sites.sum() / areas.sites.sum()  # Scalar+[F]
vorticity = curl(flux) + boundary_vorticity                # Bivector+[V]

# A streamfunction draws the conserved flux's paths on the native triangles.
streamfunction = d0.lstsq(flux)                            # Scalar+[V]
save_figure(render.draw_fields(topology, vertices, pressure, vorticity, streamfunction), "dec_stokes_fields");

The upper panel shows pressure, falling from warm to cool colours across the obstacle. The lower panel shows the two senses of vorticity, concentrated where the flow shears past the walls and obstacle. The same streamlines cross both panels; their spacing records how the throughput divides.

In [ ]:
inlet_reading = topology.reading(inlet, 1)
outlet_reading = topology.reading(outlet, 1)
throughput = -(inlet_reading * (boundary * flux)).sites.sum()
# The faces' pressures summed onto their edges: on a boundary edge, the pressure of the one face beside it.
edge_pressure = (topology.incidence[2, 1].adjugate() * pressure).sum(axis=0)  # Scalar+[E]
pressure_drop = (inlet_reading * edge_pressure).sites.mean() - (outlet_reading * edge_pressure).sites.mean()
print("Throughput per unit depth:", throughput.to_array())
print("Pressure drop near the ends:", pressure_drop.to_array())

## Flow paths

Streaks follow the computed velocity, constant on each triangle, over the pressure, each as long as the flow is fast. They run along the streamfunction's contours and pass the obstacle without crossing its boundary.

In [ ]:
# The first moment of the outward flux gives the face's mean velocity.
outward_flux = topology.boundary[2, 1] * flux              # [3] Scalar+[F]
velocity = -((corners - face_centres) * outward_flux).sum(axis=0) / (2 * areas)  # Point-[F]

save_figure(render.draw_flow(topology, vertices, pressure, velocity), "dec_stokes_flow");

## Reversing edge directions

Reversing an edge flips its signed flux reading and the incidences that refer to it. The corresponding dual edge reverses too, leaving its Hodge weight unchanged. Face balances and the vorticity bivector at each vertex therefore stay invariant under any choice of edge directions.

In [ ]:
# Reverse every third edge, together with the primal and dual readings on it.
edge_signs = mv.scalar(np.where(np.arange(E) % 3 == 0, -1, 1)[:, None]).field()  # Scalar+[E]
reorientation = spdiag(edge_signs * Scalar)                # (Scalar+ <- Scalar+)[E <- E]
vorticity_reorientation = spdiag(edge_signs * Bivector)    # (Bivector+ <- Bivector+)[E <- E]
reversed_d0_vorticity = vorticity_reorientation(d0_vorticity)  # (Bivector+ <- Bivector+)[E <- V]
reversed_d1 = d1(reorientation)                            # (Scalar+ <- Scalar+)[F <- E]
reversed_star1 = vorticity_reorientation.adjugate()(star1(reorientation))  # (Patch- <- Scalar+)[E <- E]
reversed_flux = reorientation(flux)                        # Scalar+[E]
reversed_curl = inverse_star0(reversed_d0_vorticity.adjugate()(reversed_star1))  # (Bivector+ <- Scalar+)[V <- E]

sample_edge = topology.reading(np.array([300]), 1)         # Scalar+[1 <- E]
first_vertex = topology.reading(np.array([0]), 0)          # Scalar+[1 <- V]
print("Sample edge flux before and after reversal:", (sample_edge * flux).batch().to_array(),
      (sample_edge * reversed_flux).batch().to_array())
print("Vertex 0 vorticity before and after:       ", (mv.xy.dual() & (first_vertex * vorticity)).batch().to_array(),
      (mv.xy.dual() & (first_vertex * reversed_curl(reversed_flux))).batch().to_array())

In [ ]:
# checks
potential = (mv.x & vertices).squared() + 0.3 * (mv.y & vertices)
np.testing.assert_allclose(d1(d0(potential)).kernel, 0, atol=1e-9)
np.testing.assert_allclose(d1(flux).kernel, 0, atol=1e-9)
np.testing.assert_allclose(reversed_d1(reversed_flux).kernel, 0, atol=1e-9)
np.testing.assert_allclose((reversed_curl(reversed_flux) - vorticity).kernel, 0, atol=1e-9)
np.testing.assert_allclose((vorticity & d0_vorticity.adjugate()(star1(flux))).sites.sum().kernel,
                           (star1.adjugate()(d0_vorticity(vorticity)) & flux).sites.sum().kernel, atol=1e-9)